In [3]:
import numpy as np
import pandas as pd
import os

In [4]:
df_wide_matrix = pd.read_csv("/kaggle/input/datasets/anubhashukla/normalize-and-export/temp_matrix.csv", index_col='time')
node_coords_map = pd.read_csv("/kaggle/input/datasets/anubhashukla/normalize-and-export/node_coordinates.csv")

In [5]:
# Clean column text strings to prevent trailing character mismatches
df_wide_matrix.columns = [str(c).strip() for c in df_wide_matrix.columns]
node_coords_map['Node'] = node_coords_map['Node'].astype(str).str.strip()

In [6]:
# 🔍 CRITICAL FIX 1: Explicitly grab EXACTLY the first 30 available nodes columns
target_30_nodes = list(df_wide_matrix.columns)[:30]
df_wide_matrix = df_wide_matrix[target_30_nodes]

In [7]:
# Double check that we actually have exactly 30 nodes in our list
if len(target_30_nodes) < 30:
    raise ValueError(f"❌ Core Error: Your dataset only has {len(target_30_nodes)} unique nodes. You need a larger spatial footprint!")

In [8]:
df_nodes = node_coords_map[node_coords_map['Node'].isin(target_30_nodes)].copy()
df_nodes.columns = ['Node', 'Lat', 'Lon']
# Drop any duplicate metadata definitions to ensure it stays exactly 30 rows long
df_nodes = df_nodes.drop_duplicates(subset=['Node'])
df_nodes.to_csv('nodes.csv', index=False)

In [9]:
final_long_records = []

# 🔍 CRITICAL FIX 2: Loop exactly 30 times using a strict range to guarantee dimension safety
for i in range(30):
    node_name = target_30_nodes[i]
    
    # Isolate the hourly vector array per turbine
    df_series = pd.DataFrame(df_wide_matrix[node_name]).copy()
    df_series.reset_index(inplace=True)
    df_series.columns = ['Timestamp', 'Power']
    df_series['Node'] = f"Node_{i+1:02d}" # Use clean indexed names to align tensors perfectly
    
    # Force the Power column values to be numeric floats
    df_series['Power'] = pd.to_numeric(df_series['Power'], errors='coerce')
    df_series['Power'] = df_series['Power'].fillna(0.0) 

    # 💥 NORMALIZATION: Scale your target feature variable between 0.0 and 1.0
    max_capacity = float(df_series['Power'].max())
    df_series['Power'] = df_series['Power'] / (max_capacity if max_capacity > 0 else 1.0)
    
    # 🧠 FEATURE ENGINEERING: Deriving correlated WindSpeed parameters mathematically
    df_series['WindSpeed'] = np.clip((df_series['Power'] * 13.5) + np.random.normal(0, 0.4, len(df_series)), 0.0, 25.0)
    
    final_long_records.append(df_series)

In [10]:
df_final_grid = pd.concat(final_long_records, ignore_index=True)
df_final_grid.to_csv('grid_timeseries.csv', index=False)
df_nodes['Node'] = [f"Node_{i+1:02d}" for i in range(30)]
df_nodes.to_csv('nodes.csv', index=False)